# 0. Install Dependencies

In [1]:
# Dependências já instaladas no ambiente .venv (Python 3.8)
# %pip install tensorflow==2.3.1 gym keras-rl2 gym[atari]

# 1. Test Random Environment with OpenAI Gym

In [2]:
import gym 
import random

In [3]:
env = gym.make('SpaceInvaders-v0')
height, width, channels = env.observation_space.shape
actions = env.action_space.n

In [4]:
env.unwrapped.get_action_meanings()

['NOOP', 'FIRE', 'RIGHT', 'LEFT', 'RIGHTFIRE', 'LEFTFIRE']

In [5]:
episodes = 50
for episode in range(1, episodes+1):
    state = env.reset()
    done = False
    score = 0 
    
    while not done:
        env.render()
        action = random.choice([0,1,2,3,4,5])
        n_state, reward, done, info = env.step(action)
        score+=reward
    print('Episode:{} Score:{}'.format(episode, score))
env.close()

Episode:1 Score:55.0
Episode:2 Score:125.0
Episode:3 Score:50.0
Episode:4 Score:70.0
Episode:5 Score:110.0
Episode:6 Score:30.0
Episode:7 Score:110.0
Episode:8 Score:140.0
Episode:9 Score:60.0
Episode:10 Score:395.0
Episode:11 Score:190.0
Episode:12 Score:70.0
Episode:13 Score:75.0
Episode:14 Score:100.0
Episode:15 Score:80.0
Episode:16 Score:80.0
Episode:17 Score:60.0
Episode:18 Score:75.0
Episode:19 Score:45.0
Episode:20 Score:45.0
Episode:21 Score:395.0
Episode:22 Score:110.0
Episode:23 Score:235.0
Episode:24 Score:135.0
Episode:25 Score:135.0
Episode:26 Score:295.0
Episode:27 Score:105.0
Episode:28 Score:165.0
Episode:29 Score:65.0
Episode:30 Score:120.0
Episode:31 Score:105.0
Episode:32 Score:20.0
Episode:33 Score:415.0
Episode:34 Score:155.0
Episode:35 Score:210.0
Episode:36 Score:180.0
Episode:37 Score:135.0
Episode:38 Score:190.0
Episode:39 Score:225.0
Episode:40 Score:60.0
Episode:41 Score:210.0
Episode:42 Score:135.0
Episode:43 Score:150.0
Episode:44 Score:135.0
Episode:45 Sc

# 2. Create a Deep Learning Model with Keras

In [6]:
import numpy as np
import tensorflow as tf

tf.compat.v1.disable_eager_execution()

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Convolution2D
from tensorflow.keras.optimizers import Adam

2026-10-16 17:13:43.147571: W tensorflow/stream_executor/platform/default/dso_loader.cc:59] Could not load dynamic library 'libcudart.so.10.1'; dlerror: libcudart.so.10.1: cannot open shared object file: No such file or directory
2026-10-16 17:13:43.147613: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [7]:
def build_model(height, width, channels, actions):
    model = Sequential()
    model.add(Convolution2D(32, (8,8), strides=(4,4), activation='relu', input_shape=(3,height, width, channels)))
    model.add(Convolution2D(64, (4,4), strides=(2,2), activation='relu'))
    model.add(Convolution2D(64, (3,3), activation='relu'))
    model.add(Flatten())
    model.add(Dense(512, activation='relu'))
    model.add(Dense(256, activation='relu'))
    model.add(Dense(actions, activation='linear'))
    return model

In [8]:
if 'model' in locals():
    del model

In [9]:
model = build_model(height, width, channels, actions)

2026-10-16 17:13:44.476726: W tensorflow/stream_executor/platform/default/dso_loader.cc:59] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2026-10-16 17:13:44.476755: W tensorflow/stream_executor/cuda/cuda_driver.cc:312] failed call to cuInit: UNKNOWN ERROR (303)
2026-10-16 17:13:44.476773: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (linux): /proc/driver/nvidia/version does not exist
2026-10-16 17:13:44.476972: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN)to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-16 17:13:44.484721: I tensorflow/core/platform/profile_utils/cpu_utils.cc:104] CPU Frequency: 2394240000 Hz
20

In [10]:
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 3, 51, 39, 32)     6176      
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 3, 24, 18, 64)     32832     
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 3, 22, 16, 64)     36928     
_________________________________________________________________
flatten (Flatten)            (None, 67584)             0         
_________________________________________________________________
dense (Dense)                (None, 512)               34603520  
_________________________________________________________________
dense_1 (Dense)              (None, 256)               131328    
_________________________________________________________________
dense_2 (Dense)              (None, 6)                 1

# 3. Build Agent with Keras-RL

In [11]:
from rl.agents import DQNAgent
from rl.memory import SequentialMemory
from rl.policy import LinearAnnealedPolicy, EpsGreedyQPolicy

In [12]:
def build_agent(model, actions):
    policy = LinearAnnealedPolicy(EpsGreedyQPolicy(), attr='eps', value_max=1., value_min=.1, value_test=.2, nb_steps=10000)
    memory = SequentialMemory(limit=1000, window_length=3)
    dqn = DQNAgent(model=model, memory=memory, policy=policy,
                  enable_dueling_network=True, dueling_type='avg', 
                   nb_actions=actions, nb_steps_warmup=1000
                  )
    return dqn

In [13]:
dqn = build_agent(model, actions)
dqn.compile(Adam(lr=1e-4))

ValueError: Tensor("dense_3/kernel/Read/ReadVariableOp:0", shape=(256, 7), dtype=float32) must be from the same graph as Tensor("dense_1/Relu:0", shape=(None, 256), dtype=float32) (graphs are <tensorflow.python.framework.ops.Graph object at 0x7e751f087e20> and FuncGraph(name=keras_graph, id=139041496773104)).

In [ ]:
dqn.fit(env, nb_steps=10000, visualize=False, verbose=2)

In [ ]:
scores = dqn.test(env, nb_episodes=10, visualize=True)
print(np.mean(scores.history['episode_reward']))
env.close()

# 4. Reloading Agent from Memory

In [ ]:
dqn.save_weights('SavedWeights/10k-Fast/dqn_weights.h5f', overwrite=True)

In [ ]:
del model, dqn

In [ ]:
model = build_model(height, width, channels, actions)
dqn = build_agent(model, actions)
dqn.compile(Adam(lr=1e-4))
dqn.load_weights('SavedWeights/10k-Fast/dqn_weights.h5f')